# 05 - Pretrained baselines (frozen backbones)

**What this notebook does**: trains four ImageNet-pretrained backbones with a small classification
head on the `faithful` split, so MiniConvNet can be compared against standard architectures on
identical data: `ResNet50`, `VGG16`, `MobileNetV3Small`, `EfficientNetV2B0`.

> **[CHOICE] Frozen backbones, four models, CPU scope.** The backbone is frozen
> (`trainable_base=False`) and only the head is trained. Full fine-tuning of a 23M-parameter backbone
> is not affordable on CPU, and feature extraction is the standard comparison protocol the paper's
> table uses anyway. v2's optional extended set (VGG19, InceptionV3, ConvNeXtTiny) is out of scope
> this round. State both decisions whenever these numbers are quoted.

**What must already exist**: split CSVs from notebook 00, and internet access the first time (Keras
downloads the ImageNet weights; they are cached afterwards).

**CPU cost warning**: these backbones are far heavier per epoch than MiniConvNet even with the
backbone frozen - every epoch still runs a full forward pass through ResNet50/VGG16. Each model gets
its own time estimate **and its own cell**, so a slow one can be skipped without losing the others.

**Where results go**: `outputs/results_table.csv` as canonical rows (one per model), since a single
feature-extraction run is the standard protocol. Every run is collapse-checked and saves raw
predictions.

**What "looks right"**: baselines clearly above chance within a few epochs (frozen ImageNet features
converge fast), and the familiar pattern from both previous attempts - high tumour-vs-healthy
accuracy with much weaker subtype discrimination (LESSON 11).

> **Note on EfficientNetV2B0 (section 3d).** Its first pass, trained with the same settings as the
> other three, partially collapsed - it never predicted `large.cell.carcinoma`. It is now trained
> with inverse-frequency class weights (and, if needed, a halved head learning rate). That makes its
> row **not** a like-for-like protocol comparison with the other three; the departure is recorded in
> its `notes` column and prediction metadata.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('baseline epochs:', EPOCHS_BASELINE, '| lr:', LR_BASELINE)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_utils import load_split, make_split_datasets, split_counts
from src.models import build_baseline, count_params, DEFAULT_BASELINES
from src.train_utils import (set_global_seeds, compute_report, compile_model, optimizer_summary,
                             class_weights_for, compute_class_weights, make_callbacks,
                             make_epoch_timer, save_history, plot_history, final_epoch_summary,
                             run_name_for, estimate_training_time)
from src.evaluate_utils import (predict, compute_metrics, detect_collapse, print_collapse_report,
                                plot_confusion_matrix, confusion, tumor_vs_subtype_breakdown,
                                interpret_breakdown, save_predictions, load_predictions,
                                result_row_from_metrics, record_canonical, load_results)

set_global_seeds(SEED)
for k, v in compute_report().items():
    print(f'{k}: {v}')
print()
print('baselines to train:', DEFAULT_BASELINES)

In [ ]:
SPLIT_FOR_BASELINES = 'faithful'

sdf = load_split(SPLIT_FOR_BASELINES)
# Baselines use sparse labels: label smoothing is a MiniConvNet anti-collapse
# measure (LESSON 2), not part of the baseline comparison protocol.
train_ds, val_ds, test_ds, frames = make_split_datasets(sdf)
class_weight = class_weights_for(SPLIT_FOR_BASELINES, frames['train']['label'].values)

print(split_counts(sdf).to_string())
print('class_weight:', class_weight if class_weight else 'None (by design for faithful)')

## 1. Baseline runner

`cw` and `lr` exist for the EfficientNetV2B0 remediation in section 3d and default to the
project-wide behaviour, so calling `run_baseline(name)` with no extra arguments trains exactly what
the first three baselines were trained with.

In [ ]:
SPLIT_DEFAULT = 'split_default'   # sentinel: use the split's own class-weight policy


def run_baseline(name, epochs=EPOCHS_BASELINE, trainable_base=False,
                 cw=SPLIT_DEFAULT, lr=LR_BASELINE, note_extra=''):
    """Train one frozen-backbone baseline, evaluate it, and record the result."""
    set_global_seeds(SEED)
    run_name = run_name_for(name.lower(), SPLIT_FOR_BASELINES)
    weights = class_weight if isinstance(cw, str) and cw == SPLIT_DEFAULT else cw
    print('=' * 72)
    print('baseline:', name, '| run:', run_name)

    model = build_baseline(name, trainable_base=trainable_base)
    compile_model(model, lr=lr, label_smoothing=0.0)   # sparse loss
    params = count_params(model)
    print('params      :', params)
    print('optimizer   :', optimizer_summary(model))
    print('class_weight:', {CLASS_NAMES[k]: round(v, 3) for k, v in weights.items()}
          if weights else 'None')

    timer = make_epoch_timer(verbose=0)
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                        class_weight=weights,
                        callbacks=make_callbacks(run_name, timer=timer), verbose=2)
    save_history(history, run_name, timer=timer)
    summary = final_epoch_summary(history, timer=timer)
    print('\n', summary)

    y_true, y_pred, y_prob = predict(model, test_ds)
    metrics = compute_metrics(y_true, y_pred, y_prob)
    save_predictions(run_name, y_true, y_pred, y_prob,
                     meta={'split_variant': SPLIT_FOR_BASELINES, 'baseline': name,
                           'protocol': 'frozen backbone, head-only training',
                           'class_weight': 'inverse_frequency' if weights else 'none',
                           'head_lr': lr})

    collapse = detect_collapse(history=history, kappa=metrics['cohen_kappa'],
                               mcc=metrics['mcc'], y_pred=y_pred)
    breakdown = tumor_vs_subtype_breakdown(y_true, y_pred)

    print('\ntest metrics:', {k: round(v, 4) for k, v in metrics.items()})
    print('\nconfusion matrix (an all-zero COLUMN = a class never predicted):')
    print(confusion(y_true, y_pred))
    print()
    print_collapse_report(collapse, run_name)
    print('\n' + interpret_breakdown(breakdown))

    plot_history(history, run_name)
    plot_confusion_matrix(y_true, y_pred, run_name)

    record_canonical(result_row_from_metrics(
        model_name=name, metrics=metrics, collapse=collapse,
        arch_variant='transfer_frozen' if not trainable_base else 'transfer_finetuned',
        split_variant=SPLIT_FOR_BASELINES, params=params['total_params'],
        epochs_trained=summary['epochs_trained'], n_runs=1, breakdown=breakdown,
        notes=(f"ImageNet feature extraction, head-only training; "
               f"lr={lr}; class_weight={'inverse_frequency' if weights else 'none'}; "
               f"{summary.get('mean_seconds_per_epoch')}s/epoch, "
               f"{summary.get('total_minutes')} min total on CPU{note_extra}")))

    tf.keras.backend.clear_session()
    return {'model': name, 'run_name': run_name, 'status': collapse['status'],
            'params': params['total_params'],
            'n_predicted_classes': collapse['details'].get('n_predicted_classes'),
            'binary_tumor_acc': breakdown['binary_tumor_vs_healthy_accuracy'],
            'subtype_acc': breakdown['subtype_accuracy_all_tumors'],
            'minutes': summary.get('total_minutes'), **metrics}

In [ ]:
# Verification helper: reads predictions back from the SAVED FILE on disk rather
# than from the in-memory result. The file is the evidence (LESSON 5), so the
# check has to interrogate the file.
def verify_all_classes(run_name):
    y_true, y_pred, _ = load_predictions(run_name)
    counts = np.bincount(y_pred, minlength=NUM_CLASSES)
    missing = [CLASS_NAMES[i] for i in range(NUM_CLASSES) if counts[i] == 0]
    print(f'{run_name}: {len(y_pred)} predictions read back from disk')
    for i, c in enumerate(CLASS_NAMES):
        print(f'  predicted {c:26s} {counts[i]:4d}'
              + ('   <-- NEVER PREDICTED' if not counts[i] else ''))
    print(f'\naccuracy from file: {(y_true == y_pred).mean():.4f}')
    if missing:
        print(f'STILL PARTIALLY COLLAPSED: {len(missing)} class(es) never predicted - {missing}')
    else:
        print('ALL 4 CLASSES PREDICTED - partial collapse resolved.')
    return len(missing) == 0

## 2. Time estimate for the whole baseline set

ResNet50 is timed as the representative model and the estimate is scaled by the number of baselines.
That is rough - VGG16 is slower per epoch, MobileNetV3Small much faster - so treat it as an
order-of-magnitude figure and re-check per model if it lands near the threshold.

**Read this before section 3.** If it exceeds the threshold, agree the cost first, or run a subset.

In [ ]:
est = estimate_training_time(
    model_fn=lambda: compile_model(build_baseline('ResNet50'), lr=LR_BASELINE,
                                   label_smoothing=0.0, verbose=False),
    train_ds=train_ds, val_ds=val_ds,
    planned_epochs=EPOCHS_BASELINE, n_runs=len(DEFAULT_BASELINES),
    class_weight=class_weight)
print()
print('Scaled from ResNet50 alone: VGG16 is typically slower per epoch and MobileNetV3Small much '
      'faster, so the per-model spread around this figure is wide.')

## 3. The four baselines, one cell each

Each architecture gets its own cell so a failed download, an OOM or a time overrun costs you that one
model rather than the whole notebook - and so a single model can be re-run without touching the
others.

### 3a. ResNet50

In [ ]:
results = {}
results['ResNet50'] = run_baseline('ResNet50')

### 3b. VGG16

In [ ]:
results['VGG16'] = run_baseline('VGG16')

### 3c. MobileNetV3Small

In [ ]:
results['MobileNetV3Small'] = run_baseline('MobileNetV3Small')

### 3d. EfficientNetV2B0 — with class weights (partial-collapse remediation)

**The diagnosis.** The first pass at this baseline, trained with the same settings as the other three
(no class weights, head `lr = 1e-4`), came back `INVALID_partial_collapse`. Confirmed directly from
its saved predictions rather than inferred from its metrics - it never predicted
`large.cell.carcinoma` once across all 315 test images:

```
                          pred:  adeno  large  normal  squam
adenocarcinoma                     101      0       2     17
large.cell.carcinoma                44      0       0      7    <- all 51 lost
normal                               1      0      53      0
squamous.cell.carcinoma             45      0       0     45
                                          ^^^ column never used
```

Its 0.6317 accuracy was real but unreportable: the model had dropped the smallest tumour class
outright (51 of 315 test images) and spent its capacity on the other three.

**The remediation, in two steps.** This cell is step 1:

1. **Inverse-frequency `class_weight`** - the approach already used for the imbalanced `clean` split
   (LESSON 7), applied here to `faithful` for this model only. It raises the loss contribution of
   `large.cell.carcinoma` so dropping the class is no longer cheap.
2. If that is not enough, **halve the head learning rate** (1e-4 -> 5e-5) as well - the conditional
   cell below, which is skipped entirely if step 1 worked.

`run_baseline()` overwrites this model's prediction file and replaces its row in
`results_table.csv`, so no other model is touched.

> **[CHOICE] This breaks protocol symmetry, deliberately.** The other three baselines keep the
> unweighted protocol and are **not** re-run - they trained cleanly and their rows stand. Quote the
> departure whenever the EfficientNetV2B0 row is quoted; it is recorded in the row's `notes` and in
> the prediction metadata so it cannot be lost.

In [ ]:
eff_cw = compute_class_weights(frames['train']['label'].values)
print('inverse-frequency class weights:')
for k, v in eff_cw.items():
    print(f'  {CLASS_NAMES[k]:26s} {v:.3f}')
print()

results['EfficientNetV2B0'] = run_baseline(
    'EfficientNetV2B0', cw=eff_cw,
    note_extra='; class weights added to remedy a partial collapse (large.cell.carcinoma never '
               'predicted) in the unweighted first pass')

In [ ]:
eff_fixed = verify_all_classes('efficientnetv2b0_faithful')
print('\nstatus recorded in results_table.csv:', results['EfficientNetV2B0']['status'])

In [ ]:
# Step 2, CONDITIONAL: runs only if step 1 left the collapse in place. A lower head
# learning rate gives the minority class more chances to move the decision boundary
# before the head settles. Costs nothing if step 1 already worked.
if eff_fixed:
    print('Step 1 (class weights) resolved it - skipping the learning-rate step, nothing to do.')
else:
    print(f'Step 1 was not enough. Re-running with head lr {LR_BASELINE} -> {LR_BASELINE / 2}, '
          'class weights kept on.\n')
    results['EfficientNetV2B0'] = run_baseline(
        'EfficientNetV2B0', cw=eff_cw, lr=LR_BASELINE / 2,
        note_extra='; class weights + halved head lr, to remedy a partial collapse '
                   '(large.cell.carcinoma never predicted) that class weights alone did not fix')
    eff_fixed = verify_all_classes('efficientnetv2b0_faithful')
    if not eff_fixed:
        print('\nBoth remediation steps failed. Report this as a finding with the confusion matrix '
              'above - do not keep trying settings until something sticks.')

## 4. Baseline summary

**Looks right**: four rows, all `ok`, all with `n_predicted_classes = 4`, and all with far more
parameters than MiniConvNet's ~499K - that parameter gap is the entire point of the comparison. Note
the `binary_tumor_acc` vs `subtype_acc` columns: the pattern that held across both previous attempts
is strong detection with much weaker subtype discrimination (LESSON 11).

**Caveat to carry into the write-up**: EfficientNetV2B0 alone was trained with class weights (and
possibly a halved head learning rate) after its unweighted first pass partially collapsed, so its row
is not a like-for-like protocol comparison with the other three.

In [ ]:
tbl = pd.DataFrame(results.values())
print(tbl[['model', 'params', 'accuracy', 'f1_macro', 'cohen_kappa', 'binary_tumor_acc',
           'subtype_acc', 'n_predicted_classes', 'minutes', 'status']].round(4).to_string(index=False))

bad = tbl[tbl['status'] != VALID_TAG]
print('\ninvalid baselines:', bad['model'].tolist() if len(bad) else 'none')
print('total baseline wall clock:', round(tbl['minutes'].fillna(0).sum(), 1), 'min')

In [ ]:
canon = load_results('canonical')
print(canon[['model', 'arch_variant', 'params', 'accuracy', 'accuracy_std', 'f1_macro',
             'binary_tumor_acc', 'subtype_acc', 'n_runs', 'status']].round(4).to_string(index=False))
print('\nnext: 06_evaluate_and_compare.ipynb')